# 🎙️ Bahgat voice training — RVC (primary) + GPT-SoVITS (fallback)

**Goal:** train a voice model of Bahgat Saber from ~10–30 min of clean, single-speaker
Egyptian-Arabic speech, so the Telegram bot can speak in his voice.

**Run order (top to bottom, one pass):**
1. §0 GPU check
2. §1 RVC setup (clone + install + pretrained weights)
3. §2 Dataset from Google Drive (`bahgat_dataset.zip`)
4. §3 RVC training (**primary** — do this first)
5. §4 GPT-SoVITS training (**fallback** — only if RVC fails or sounds wrong)
6. §5 Export trained weights back to Drive
7. §6 What to do next (wiring the weights into the bot)

**Primary = RVC.** Reasons: RVC is language-agnostic (it converts audio → audio, so
Egyptian Arabic needs no text frontend), it is the best-documented route for a 10+ min
dataset, and its output (`.pth` + `.index`) plugs straight into the bot's planned
pipeline (Edge TTS → RVC conversion). GPT-SoVITS is a TTS model: it needs text
transcripts *and* Arabic is **not** in its officially supported language list
(en/ja/ko/zh/yue) — see §4 for the caveat before you spend GPU time on it.

**Before you run anything:** open **Runtime → Change runtime type → Hardware
accelerator → T4 GPU**, then **Runtime → Run all** (or run cells one by one).
Keep the browser tab open while training runs — closing it kills the session.

## §0 — GPU check

Free Colab gives you an **NVIDIA T4 (15 GB usable VRAM)** *when one is available* —
it is **not guaranteed**. If this cell shows no GPU, wait and retry later
(off-peak hours work best). Do not start training on a CPU runtime: it would
take 8–12 hours instead of ~1–2.

In [ ]:
!nvidia-smi
import sys, torch
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("NO GPU — stop here. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## §1 — RVC setup (primary)

* Repo: **RVC-Project/Retrieval-based-Voice-Conversion-WebUI** (`main` branch,
  pinned to a verified commit below)
* Why this repo and not a fork: it is the official, actively maintained repo
  (2026 release 2.3.x, Python 3.12, torch 2.7.1+cu128). The popular community
  fork Applio has publicly moved to maintenance mode, and the old
  Rejekts/ardha27 Colab notebooks are broken on current Colab (Python 3.12 /
  numpy 2.x / fairseq no longer installs). Current main no longer needs fairseq
  (ContentVec is loaded via Hugging Face transformers).
* **Headless CLI only.** Colab blocks the RVC Gradio WebUI on the free tier, so
  every step below uses the documented CLI (`docs/en/cli.md`). No GUI cell.

In [ ]:
# --- RVC clone (pinned commit) -------------------------------------------
# Pinned to 81eed5e8f68b6bed1789f682fe78cdd324495afc: the commit the official
# CLI docs (docs/en/cli.md) were verified against in Nov 2026.
# If `git checkout` fails because the repo moved on, delete that line and use main.
!apt-get update -qq && apt-get install -y -qq ffmpeg unzip > /dev/null 2>&1
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/rvc
%cd /content/rvc
!git checkout 81eed5e8f68b6bed1789f682fe78cdd324495afc
!git log --oneline -1
!python --version

In [ ]:
# --- RVC dependencies -------------------------------------------------------
# Colab already ships torch + CUDA. The repo's requirements file really is
# spelled "requirments_cu128_py312.txt" (missing an 'e') upstream.
%cd /content/rvc
!pip install -q huggingface_hub faiss-cpu soundfile
!ls requirments*.txt requirements*.txt 2>/dev/null
!grep -i fairseq requirments_cu128_py312.txt || echo "no fairseq in requirements (good — current main uses transformers ContentVec)"
!pip install -q -r requirments_cu128_py312.txt || pip install -q -r requirements.txt
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- RVC pretrained assets (download once per session) ----------------------
# hubert_base  = ContentVec features (16 kHz)   |  rmvpe.pt = pitch model
# pretrained_v2/f0G40k.pth + f0D40k.pth = v2 40k base generator/discriminator
# (the base model was pretrained on ~50 h of VCTK; we only fine-tune it).
%cd /content/rvc
!hf download lj1995/VoiceConversionWebUI --revision main --include "hubert_base/*" --local-dir assets
!hf download lj1995/VoiceConversionWebUI rmvpe.pt --revision main --local-dir assets/rmvpe
!hf download lj1995/VoiceConversionWebUI --revision main --include "pretrained_v2/f0G40k.pth" "pretrained_v2/f0D40k.pth" --local-dir assets
!ls -lh assets/hubert_base/ assets/rmvpe/ assets/pretrained_v2/

## §2 — Dataset from Google Drive

**Expected file:** `bahgat_dataset.zip` in the **root of your Google Drive**
("My Drive", not inside a folder). The clean job builds it as a zip of:

```
bahgat_dataset.zip
├── clean/bahgat_0001.wav ... bahgat_XXXX.wav   (16 kHz mono, 3–10 s each)
├── clean/filelist.txt                          (one relative path per line)
└── transcripts/bahgat_0001.txt ...             (one .txt per clip, same basename)
```

The notebook also accepts the zip with `training/clean/` + `training/transcripts/`
prefixes, or a single combined transcripts JSON (`{basename: text}`) instead of
per-clip `.txt` files. A `.srt` alone cannot be mapped to clips — convert it to
per-clip `.txt` (or the JSON) first.

In [ ]:
# --- Mount Drive + fetch the dataset zip ------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os
ZIP = "/content/drive/MyDrive/bahgat_dataset.zip"
assert os.path.exists(ZIP), "bahgat_dataset.zip NOT FOUND in Drive root — upload it to My Drive first."
print(f"zip found: {round(os.path.getsize(ZIP)/1e6, 1)} MB")
!mkdir -p /content/dataset && unzip -o -q /content/drive/MyDrive/bahgat_dataset.zip -d /content/dataset
!find /content/dataset -type f | head -20
print("wav files:", len(__import__('glob').glob('/content/dataset/**/*.wav', recursive=True)))

In [ ]:
# --- Dataset audit: resample check + stats ----------------------------------
# Contract: 16 kHz mono WAV, 3–10 s per clip, Bahgat speaking alone.
# RVC's preprocess resamples to 40 kHz itself, so 16 kHz input is fine —
# but anything that is not mono / is wildly off-length gets flagged here.
import glob, os
import soundfile as sf

wavs = sorted(glob.glob("/content/dataset/**/*.wav", recursive=True))
assert wavs, "no .wav files found in the zip — check the zip layout (see §2)."
total, bad = 0.0, []
for w in wavs:
    info = sf.info(w)
    total += info.duration
    if info.samplerate != 16000 or info.channels != 1 or not (3 <= info.duration <= 10):
        bad.append((os.path.basename(w), info.samplerate, info.channels, round(info.duration, 1)))
print(f"clips: {len(wavs)} | total clean audio: {total/60:.1f} minutes")
print(f"target: 10+ min (10–30 min ideal). {'OK' if total >= 600 else 'BELOW 10 MIN — RVC quality will suffer; add more audio.'}")
print("clips outside contract (not 16kHz mono 3-10s):", len(bad))
for b in bad[:20]:
    print("   ", b)

In [ ]:
# --- Build the RVC training set + hold out 5 clips for listening tests -------
# RVC has NO train/val split: it trains on everything you give it and you judge
# quality by ear. So we hold out 5 clips by hand (never fed to training) and use
# them later for A/B listening tests of the finished model.
import glob, os, random, shutil
random.seed(7)
wavs = sorted(glob.glob("/content/dataset/**/*.wav", recursive=True))
holdout = random.sample(wavs, min(5, len(wavs)))
os.makedirs("/content/rvc_dataset", exist_ok=True)
os.makedirs("/content/holdout", exist_ok=True)
for w in wavs:
    if w in holdout:
        shutil.copy(w, "/content/holdout/")
    else:
        shutil.copy(w, "/content/rvc_dataset/")
print(f"training clips: {len(os.listdir('/content/rvc_dataset'))} | holdout clips: {len(os.listdir('/content/holdout'))}")
print("holdout:", [os.path.basename(h) for h in holdout])

## §3 — RVC training (PRIMARY)

### Hyperparameters — and why these values

| Setting | Value | Why |
|---|---|---|
| Version | **v2** | Current standard; v1 is legacy, v3 weights were never shipped |
| Sample rate | **40 kHz** | The default almost every community v2 checkpoint uses. **48 kHz** needs ~24 GB VRAM (not a T4) and buys nothing audible for Telegram voice notes (Opus-compressed) — and our source clips are 16 kHz, so upsampling to 48 k adds no information. 32 kHz is the budget option if VRAM is tight |
| F0 | on, **RMVPE on CUDA** | Pitch guidance is what makes a speaking voice (not just singing) sound like the person. RMVPE is the best-quality F0 extractor and is GPU-accelerated; `harvest`/`pm` are CPU-only and worse |
| Batch size | **4** | WebUI default rule is VRAM_GB / 2 (≈ 7 on a T4), but 4 is safer on a shared Colab T4 and slightly more accurate. Drop to 2 only on CUDA out-of-memory |
| Epochs | **150** | Community guidance is 100–300; more data = fewer epochs needed. 100 risks underfitting an expressive/shouty voice, 300+ on ~15 min risks memorisation. 150 is the sweet spot for ~15 min |
| Save every | **10 epochs** | Lets you pick the best checkpoint by ear instead of only the last one |
| `-sw 1` | on | Also saves the small inference-ready `.pth` (`assets/weights/bahgat.pth`) |
| `-l 0` / `-c 0` | off | Keep all checkpoints (needed for resume); don't cache the dataset in GPU memory (T4 VRAM is precious) |
| Speaker | `bahgat`, id 0 | Single-speaker model |

**Expected time on a free T4 (~15 min audio):** setup ~10–15 min, preprocess
~2–5 min, RMVPE F0 ~3–5 min, HuBERT ~2–4 min, training 150 epochs ~45–75 min,
index ~2–5 min. **Total ≈ 1.5–2 hours** — inside one 12 h Colab session.

In [ ]:
# --- Step 1: preprocess (resample to 40k, slice, write logs/bahgat/filelist.txt)
# Args: dataset_dir sample_rate workers exp_dir disable_multiprocessing slice_param
%cd /content/rvc
!python train/preprocess.py /content/rvc_dataset 40000 4 logs/bahgat False 3.7
!ls logs/bahgat/ && wc -l logs/bahgat/filelist.txt && head -3 logs/bahgat/filelist.txt

In [ ]:
# --- Step 2: F0 extraction with RMVPE on CUDA --------------------------------
# Args: device nproc rank gpu_id exp_dir half_precision
%cd /content/rvc
!python train/dataset/extract_f0.py cuda 1 0 0 logs/bahgat true

In [ ]:
# --- Step 3: HuBERT (ContentVec) feature extraction ---------------------------
%cd /content/rvc
!python train/dataset/extract_hubert_feature.py cuda:0 1 0 0 logs/bahgat v2 true
!ls logs/bahgat/3-feature768/ | head -3; ls logs/bahgat/3-feature768/ | wc -l

In [ ]:
# --- Step 4: TRAIN (150 epochs, batch 4) — the long cell, ~45–75 min ----------
# Keep this tab open. Loss prints to the cell and to logs/bahgat/train.log.
# If Colab disconnects, see §6 "Resume" — checkpoints are saved every 10 epochs.
%cd /content/rvc
!python train/train.py -e bahgat -sr 40k -f0 1 -bs 4 -g 0 -te 150 -se 10 \
  -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth \
  -l 0 -c 0 -sw 1 -v v2

In [ ]:
# --- Step 5: build the retrieval index (.index) -------------------------------
# The .index makes converted speech sound closer to Bahgat (retrieval over his
# real features). The model still works without it (index_rate=0), but ship it.
# The index script's path moved between repo versions, so locate it dynamically.
%cd /content/rvc
import glob, subprocess, sys
cands = (glob.glob("infer/modules/train/extract/*index*.py")
         + glob.glob("train/*index*.py") + glob.glob("tools/*index*.py"))
print("index script candidates:", cands)
if cands:
    subprocess.run([sys.executable, cands[0], "logs/bahgat", "v2"], check=True)
else:
    print("NO index script found in this checkout — run: find . -name '*index*.py' -path '*train*'"
          " and re-run that script as: python <script> logs/bahgat v2")

In [ ]:
# --- Verify RVC outputs --------------------------------------------------------
import glob
for p in sorted(glob.glob("assets/weights/*.pth") + glob.glob("logs/bahgat/*.pth")
                + glob.glob("logs/bahgat/*.index") + glob.glob("assets/indices/*.index")):
    import os
    print(f"{round(os.path.getsize(p)/1e6, 1):>7} MB  {p}")
print()
print("Expect: assets/weights/bahgat.pth (~70-80 MB) + logs/bahgat/added_*.index")
print("If the .pth is missing, check logs/bahgat/train.log for the error.")

## §4 — GPT-SoVITS training (FALLBACK — only if RVC fails)

* Repo: **RVC-Boss/GPT-SoVITS** (`main` branch). The repo ships its own official
  Colab notebook (`Colab-WebUI.ipynb`) — if the cells below fight you, use that
  notebook instead; it tunnels the Gradio WebUI.
* Version: **v2** (not v4). v2 fine-tuning needs **6 GB min / 8 GB recommended**
  VRAM → fits a free T4. v3/v4 need **8 GB min / 16 GB recommended** → marginal
  on a 15 GB T4. v2 outputs 32 kHz audio, fine for Telegram voice notes.
* Data needed: far less than RVC — GPT-SoVITS fine-tunes from ~1 min, and our
  10–30 min is plenty. But it **needs the transcripts** (RVC does not).
* ⚠️ **Arabic caveat — read before training:** GPT-SoVITS's officially supported
  languages are English, Japanese, Korean, Chinese and Cantonese. **Arabic is not
  on that list**, so its text frontend may not handle Arabic text correctly.
  The `.list` builder below tags clips `ar`; if the pipeline rejects `ar`,
  change `LANG = "ar"` to `LANG = "en"` and re-run — the acoustic model still
  learns Bahgat's voice, only the text handling changes. **Test zero-shot
  inference with an Arabic sentence before committing to a full fine-tune.**
  This is the main reason GPT-SoVITS is the fallback, not the primary.
* Two models are trained, in this order: **SoVITS first (slow, ~30–45 min)**,
  then **GPT (fast, ~10–20 min)**. They are independent — either order works.

In [ ]:
# --- GPT-SoVITS clone + install ----------------------------------------------
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/gpt-sovits
%cd /content/gpt-sovits
!git log --oneline -1
!pip install -q -r requirements.txt
!pip install -q -r extra-req.txt --no-deps || echo "extra-req skipped (ok)"

In [ ]:
# --- GPT-SoVITS v2 pretrained weights -----------------------------------------
# Filenames as published by RVC-Boss on Hugging Face. If one 404s, check the
# repo README "pretrained models" section for the current names and adjust.
%cd /content/gpt-sovits
from huggingface_hub import snapshot_download
try:
    snapshot_download(repo_id="lj1995/GPT-SoVITS", allow_patterns=[
        "GPT_SoVITS/pretrained_models/s1bert25hz-5kh-longer-epoch=12e-step=369668.ckpt",
        "GPT_SoVITS/pretrained_models/s2Gv2Pro.14000.pth",
        "GPT_SoVITS/pretrained_models/s2Dv2Pro.14000.pth",
        "GPT_SoVITS/pretrained_models/chinese-hubert-base/*",
        "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/*",
    ], local_dir=".")
    print("download ok")
except Exception as e:
    print("DOWNLOAD FAILED:", e)
    print("Fallback: open the repo's official Colab-WebUI.ipynb — it downloads these itself.")
!find GPT_SoVITS/pretrained_models -maxdepth 2 -type f 2>/dev/null | head -30

In [ ]:
# --- Build the GPT-SoVITS .list file ------------------------------------------
# Format per line: wav_path|speaker|language|transcript
# Transcripts: per-clip .txt preferred; else one combined JSON {basename: text}.
import glob, os, json
wavs = sorted(glob.glob("/content/rvc_dataset/**/*.wav", recursive=True))
LANG = "ar"   # see §4 Arabic caveat — switch to "en" if the pipeline rejects "ar"
texts = {}
for w in wavs:
    base = os.path.splitext(w)[0]
    if os.path.exists(base + ".txt"):
        texts[w] = open(base + ".txt", encoding="utf-8").read().strip()
# Fallback: combined transcripts JSON anywhere in the dataset
if len(texts) < len(wavs) * 0.5:
    for jf in glob.glob("/content/dataset/**/*.json", recursive=True):
        try:
            d = json.load(open(jf, encoding="utf-8"))
        except Exception:
            continue
        if isinstance(d, dict):
            for w in wavs:
                key = os.path.splitext(os.path.basename(w))[0]
                if w not in texts and key in d:
                    texts[w] = str(d[key]).strip()
            print(f"filled transcripts from {jf}")
rows, missing = [], []
for w in wavs:
    t = texts.get(w, "").replace("\n", " ").replace("|", " ").strip()
    if t:
        rows.append(f"{w}|bahgat|{LANG}|{t}")
    else:
        missing.append(os.path.basename(w))
print(f"rows: {len(rows)} | clips with no transcript (skipped): {len(missing)}")
assert rows, "no transcripts found — GPT-SoVITS cannot train without them."
os.makedirs("/content/gpt-sovits/TEMP", exist_ok=True)
open("/content/gpt-sovits/TEMP/bahgat.list", "w", encoding="utf-8").write("\n".join(rows) + "\n")
print(open("/content/gpt-sovits/TEMP/bahgat.list", encoding="utf-8").read()[:400])

In [ ]:
# --- GPT-SoVITS dataset prep: text -> HuBERT -> semantic ----------------------
# These three scripts are the documented pipeline (1-get-text, 2-get-hubert-wav32k,
# 3-get-semantic). Flags can drift between versions — the --help dump below shows
# the real ones for this checkout; adjust the run commands if needed.
%cd /content/gpt-sovits
import subprocess, sys
for s in ["1-get-text.py", "2-get-hubert-wav32k.py", "3-get-semantic.py"]:
    h = subprocess.run([sys.executable, f"GPT_SoVITS/prepare_datasets/{s}", "--help"],
                       capture_output=True, text=True)
    print("###", s); print((h.stdout + h.stderr)[:600])

In [ ]:
# --- Run the three prep scripts (uses TEMP/bahgat.list from the cell above) ----
%cd /content/gpt-sovits
!python GPT_SoVITS/prepare_datasets/1-get-text.py --exp_name bahgat
!python GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py --exp_name bahgat
!python GPT_SoVITS/prepare_datasets/3-get-semantic.py --exp_name bahgat
!ls logs/bahgat/ 2>/dev/null; ls TEMP/ | head

In [ ]:
# --- GPT-SoVITS training configs (fill templates with the Bahgat values) -------
# Mirrors what the WebUI "1B fine-tune" tab does: copy the stock configs,
# point them at TEMP/bahgat.list, set epochs/batch for a T4, write TEMP/tmp_*.
# SoVITS: 8 epochs, batch 4 | GPT: 15 epochs, batch 4 (WebUI defaults).
%cd /content/gpt-sovits
import json, glob, os, yaml

def find(patterns):
    for p in patterns:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    return None

s1_ckpt = find(["GPT_SoVITS/pretrained_models/**/s1bert25hz*.ckpt"])
s2g = find(["GPT_SoVITS/pretrained_models/**/s2G*.pth"])
s2d = find(["GPT_SoVITS/pretrained_models/**/s2D*.pth"])
print("s1 ckpt:", s1_ckpt); print("s2G:", s2g); print("s2D:", s2d)
assert s1_ckpt and s2g and s2d, "pretrained weights missing — re-run the download cell."

# --- SoVITS config (s2) ---
s2 = json.load(open("GPT_SoVITS/configs/s2.json"))
s2["train"]["batch_size"] = 4
s2["train"]["epochs"] = 8
s2["train"]["save_every_epoch"] = 2
s2["data"]["training_files"] = "TEMP/bahgat.list"
s2["data"]["validation_files"] = "TEMP/bahgat.list"
s2["model_dir"] = "logs/bahgat"
s2["train"]["pretrained_s2G"] = s2g
s2["train"]["pretrained_s2D"] = s2d
os.makedirs("TEMP", exist_ok=True)
json.dump(s2, open("TEMP/tmp_s2.json", "w"), indent=2)
print("tmp_s2.json written: batch 4, epochs 8, save every 2")

# --- GPT config (s1) ---
s1_path = "GPT_SoVITS/configs/s1longer.yaml"
s1 = yaml.safe_load(open(s1_path))
s1["train"]["batch_size"] = 4
s1["train"]["epochs"] = 15
s1["train"]["pretrained_s1"] = s1_ckpt
s1["train"]["exp_name"] = "bahgat"
yaml.safe_dump(s1, open("TEMP/tmp_s1.yaml", "w"))
print("tmp_s1.yaml written: batch 4, epochs 15")
print("NOTE: if a key above does not exist in this checkout, the cell errors —")
print("print the config (json.load/yaml.safe_load) and adjust the key names.")

In [ ]:
# --- Train SoVITS first (slow, ~30-45 min), then GPT (fast, ~10-20 min) -------
# If a script rejects a flag, run `python GPT_SoVITS/s2_train.py --help`
# (or s1_train.py --help) and adjust — the flags below are the documented ones.
%cd /content/gpt-sovits
!python GPT_SoVITS/s2_train.py --config TEMP/tmp_s2.json
!python GPT_SoVITS/s1_train.py --config_file TEMP/tmp_s1.yaml

In [ ]:
# --- Verify GPT-SoVITS outputs -------------------------------------------------
import glob, os
for p in sorted(glob.glob("logs/bahgat/**/*.ckpt", recursive=True)
                + glob.glob("logs/bahgat/**/*.pth", recursive=True)):
    print(f"{round(os.path.getsize(p)/1e6, 1):>7} MB  {p}")
print()
print("Expect: logs/bahgat/ckpt/s2G*.pth + s2D*.pth (SoVITS)")
print("     +  logs/bahgat/ckpt/s1bert25hz-*-e15.ckpt (GPT)")

## §5 — Export trained weights to Drive

Everything so far lives on the **ephemeral Colab VM disk** — it is wiped when the
session ends. These cells copy **only the finished weights** to Drive
(`My Drive/bahgat_voice_models/`). Checkpoints stay on the VM unless you run the
optional backup cell (§6) — a full `logs/` folder is several GB and would eat
your 15 GB free Drive quota.

In [ ]:
# --- Export RVC weights to Drive ----------------------------------------------
import glob, os, zipfile, datetime
DATE = datetime.date.today().strftime("%Y%m%d")
OUT = "/content/drive/MyDrive/bahgat_voice_models"
os.makedirs(OUT, exist_ok=True)
files = glob.glob("/content/rvc/assets/weights/bahgat.pth") \
      + glob.glob("/content/rvc/logs/bahgat/added*.index") \
      + glob.glob("/content/rvc/logs/bahgat/*.index")
assert files, "no RVC weights found — did training finish?"
zpath = f"{OUT}/bahgat_rvc_v2_40k_{DATE}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("README.txt",
        "Bahgat RVC v2 40k\nModel: bahgat.pth (small inference weights, -sw 1)\n"
        "Index: added_*.index (FAISS retrieval index)\n"
        "Trained: 150 epochs, batch 4, RMVPE f0, from ~15 min clean 16kHz audio\n"
        "Use: Edge TTS ar-EG-ShakirNeural audio -> RVC convert (see bot docs/voice-plan.md)\n")
    for f in files:
        z.write(f, os.path.basename(f))
        print(f"added {os.path.basename(f)} ({round(os.path.getsize(f)/1e6, 1)} MB)")
print("saved:", zpath, f"({round(os.path.getsize(zpath)/1e6, 1)} MB)")

In [ ]:
# --- Export GPT-SoVITS weights to Drive (only if you ran §4) -------------------
import glob, os, zipfile, datetime
DATE = datetime.date.today().strftime("%Y%m%d")
OUT = "/content/drive/MyDrive/bahgat_voice_models"
os.makedirs(OUT, exist_ok=True)
files = glob.glob("/content/gpt-sovits/logs/bahgat/ckpt/*.pth") \
      + glob.glob("/content/gpt-sovits/logs/bahgat/ckpt/*.ckpt")
if not files:
    print("no GPT-SoVITS weights found — skip this cell if you only trained RVC.")
else:
    zpath = f"{OUT}/bahgat_gpt-sovits_v2_{DATE}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("README.txt",
            "Bahgat GPT-SoVITS v2\nSoVITS: 8 epochs, batch 4 | GPT: 15 epochs, batch 4\n"
            "Language tag used: see TEMP/bahgat.list (ar, or en if ar was rejected)\n"
            "Use: text -> GPT-SoVITS TTS (fallback if RVC is not used)\n")
        for f in files:
            z.write(f, os.path.basename(f))
            print(f"added {os.path.basename(f)} ({round(os.path.getsize(f)/1e6, 1)} MB)")
    print("saved:", zpath, f"({round(os.path.getsize(zpath)/1e6, 1)} MB)")

## §6 — Resume, troubleshooting, limits

### Resume after a disconnect (RVC)
1. Start a **new** Colab session, re-run §0–§2 (GPU check, clone+install,
   pretrained download, dataset unzip) — about 15 min.
2. Copy the backed-up checkpoints back (run the backup cell below *before*
   you lose the session, or after every 50 epochs):
   `cp /content/drive/MyDrive/bahgat_voice_models/checkpoints_rvc/G_*.pth /content/rvc/logs/bahgat/`
   (same for `D_*.pth`).
3. Re-run the training cell with the **latest checkpoint as the pretrained
   weights** and fewer total epochs, e.g. if you stopped at epoch 80:
   `python train/train.py -e bahgat -sr 40k -f0 1 -bs 4 -g 0 -te 70 -se 10 -pg logs/bahgat/G_80.pth -pd logs/bahgat/D_80.pth -l 0 -c 0 -sw 1 -v v2`
   (F0/HuBERT extraction do **not** need re-running — but they live on the wiped
   VM disk, so re-run preprocess + extract cells first; they take ~10 min.)
4. GPT-SoVITS resumes the same way: its checkpoints are in `logs/bahgat/ckpt/`;
   point the tmp configs at the latest checkpoint and re-run.

### Free-tier limits that will bite (Google publishes almost none of these exactly)
* **12 h max session** (free tier), then the VM is deleted — export to Drive
  before that.
* **Idle disconnect ~90 min** (unpublished, varies) — a running training cell
  counts as active, but a finished cell with no interaction does not. Keep the
  tab open; do not rely on background execution (Pro+ only).
* **GPU not guaranteed** — at peak times you may get CPU-only or no GPU at all.
  Retry later / off-peak. Weekly GPU hours are rationed dynamically (~15–30 h
  reported, unpublished).
* **VM disk is ephemeral (~35–78 GB)** — everything not on Drive disappears.
* **Drive is 15 GB free** (shared with Gmail/Photos) — export only finished
  weights, not full `logs/` folders.

### Common errors
* `CUDA out of memory` → batch 4 → 2 (`-bs 2`), and make sure `-c 0`.
* `ModuleNotFoundError: av` / fairseq errors → you are on an old notebook or
  checkout; this notebook uses current main, which needs neither.
* RVC WebUI won't open / Gradio blocked → expected on free Colab; use the CLI
  cells here, never the GUI cell.
* GPT-SoVITS rejects language `ar` → set `LANG = "en"` in the `.list` cell
  and re-run §4 (see the Arabic caveat).
* `hf download` 404 on a pretrained file → the filename changed upstream;
  check the repo README and adjust the `allow_patterns` / `--include` lists.

In [ ]:
# --- OPTIONAL: back up RVC checkpoints to Drive (for resume) -------------------
# Run this after every ~50 epochs, or before closing the session.
# Each G+D pair is ~200-400 MB; Drive free quota is 15 GB — keep only the latest 2 pairs.
!mkdir -p /content/drive/MyDrive/bahgat_voice_models/checkpoints_rvc
!cp /content/rvc/logs/bahgat/G_*.pth /content/rvc/logs/bahgat/D_*.pth /content/drive/MyDrive/bahgat_voice_models/checkpoints_rvc/ 2>/dev/null
!ls -lh /content/drive/MyDrive/bahgat_voice_models/checkpoints_rvc/ 2>/dev/null

## §7 — What to do next (after the weights are on Drive)

1. **Download the zip(s)** from `My Drive/bahgat_voice_models/` to your computer:
   * `bahgat_rvc_v2_40k_YYYYMMDD.zip` — contains `bahgat.pth` + `added_*.index`
   * `bahgat_gpt-sovits_v2_YYYYMMDD.zip` — fallback weights (only if you ran §4)
2. **Listening test (do this before wiring anything into the bot):** convert one
   of the 5 holdout clips (`/content/holdout/`, never trained on) and a fresh
   Edge TTS sample through the model. If it sounds like Bahgat, ship it; if it
   sounds robotic or like the wrong person, train longer (more epochs) or check
   the dataset for noisy clips.
3. **Wiring into the bot** (per `docs/voice-plan.md`, v3 — a separate task, not
   this notebook): the bot keeps generating speech with **Edge TTS
   `ar-EG-ShakirNeural`** (same text, same prosody as today), then pipes that
   audio through **RVC conversion** with `bahgat.pth` + the `.index`
   (f0 method RMVPE, index_rate ≈ 0.75). RVC runs on the bot host, so the host
   needs the RVC runtime + the two files — the notebook does not deploy them.
   GPT-SoVITS, if used instead, replaces the Edge TTS step entirely
   (text → GPT-SoVITS TTS in Bahgat's voice).
4. **Keep the parody label:** the bot stays labelled a parody bot in its bio,
   not the real Bahgat Saber.
5. **If RVC sounds wrong:** first suspect the dataset (background music, other
   speakers, shouting clipped audio), not the hyperparameters. Clean data beats
   more epochs.

*Notebook prepared 2026-11-26 for the Bahgat bot project
(`~/workspace/bahgat-bot/training/colab/`). Companion files in the same folder:
`rvc_config.md`, `gpt-sovits_config.md`, `README.md`.*